# 04b 경량 모델 크기·속도 측정

- 담당: **④ 박유정 (경량화·통합)** (이 파일은 담당자만 수정)
- 기록 파일: `experiments/log_P_light.csv`

### 할 일
- [ ] 피처 수 축소, 작은 트리, 선형 모델(로지스틱 회귀 등)로 경량 모델 만들기
- [ ] 모델 파일 크기(MB), 추론 시간(ms) 측정 (같은 장비, 워밍업 후 반복)
- [ ] 고성능 모델과 성능·크기·속도 비교표

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/data


In [2]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

OS: macOS-27.0.1-arm64-arm-64bit-Mach-O | python 3.13.18 | pandas 3.1.1 | scikit-learn 1.9.1


## 1. 데이터

In [3]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

train (6201, 4386) | test (2546, 4385) | classes 26


## 2. 측정 준비 — 크기·속도 측정 함수
모델 파일 크기(MB)와 예측 시간(ms)을 같은 컴퓨터에서 재요. 첫 실행은 느려서 한 번 미리 돌린 뒤(워밍업) 여러 번 재서 가운데 값을 써요.

In [4]:
import io, time, joblib
from scipy import sparse
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# 측정 환경 (같은 컴퓨터에서 잰 값끼리만 비교)
print("측정 환경:", platform.platform(), "| python", platform.python_version())

def measure(model, X_sample, n_repeat=20):
    """모델 크기(MB)와 예측 시간(ms). 워밍업 1회 후 반복 측정의 중앙값."""
    buffer = io.BytesIO()
    joblib.dump(model, buffer)                        # 파일로 저장했을 때의 크기
    size_mb = buffer.getbuffer().nbytes / 1024**2

    one = X_sample.iloc[:1]
    model.predict_proba(one)                          # 워밍업
    t_one = []
    for _ in range(n_repeat):
        start = time.perf_counter()
        model.predict_proba(one)
        t_one.append((time.perf_counter() - start) * 1000)

    t_all = []
    for _ in range(5):
        start = time.perf_counter()
        model.predict_proba(X_sample)
        t_all.append((time.perf_counter() - start) * 1000)

    return {"파일 크기(MB)": round(size_mb, 2),
            "1명 예측(ms)": round(float(np.median(t_one)), 2),
            f"{len(X_sample)}명 예측(ms)": round(float(np.median(t_all)), 1)}

X_timing = X.iloc[:500]   # 시간 측정용 샘플 (train에서 가져옴, test는 사용 안 함)
print("측정 함수 준비 완료")

측정 환경: macOS-27.0.1-arm64-arm-64bit-Mach-O | python 3.13.18
측정 함수 준비 완료


## 3. 경량 모델 — 로지스틱 회귀 (04a_logreg_baseline과 같은 설정)
기본 피처(0/1) + log(1+변이 개수), C=0.1, class_weight=balanced. train 전체로 학습한 뒤 크기·속도를 재요.

In [5]:
def binary_plus_count(X):
    """기본 피처(0/1) + log(1+변이 개수). 04a_logreg_baseline과 동일."""
    b = binarize_mutations(X).astype(np.float32)
    cnt = np.log1p(b.sum(axis=1).to_numpy())[:, None]
    return sparse.hstack([sparse.csr_matrix(b.to_numpy()),
                          sparse.csr_matrix(cnt)]).tocsr()

LR_PARAMS = dict(C=0.1, max_iter=2000, class_weight="balanced")

light_model = Pipeline([
    ("features", FunctionTransformer(binary_plus_count)),
    ("model", LogisticRegression(**LR_PARAMS)),
]).fit(X, y)

light_result = measure(light_model, X_timing)
light_result


{'파일 크기(MB)': 0.47, '1명 예측(ms)': 307.57, '500명 예측(ms)': 416.0}

## 4. 고성능 모델 — ③ 최신 XGBoost (03_models의 xgb_group_counts_1109_derived)
0/1 + 변이 개수 + 변이 유형별 개수(총 4,390열), 나무 300개·깊이 5.
측정 조건: 이 컴퓨터의 CPU (서비스 시나리오의 병원 PC 가정)

In [6]:
from src.features import mutation_count, mutation_type_counts

def derived_features(X):
    """③ 최신 피처: 0/1 + 변이 개수 + 변이 유형별 개수 (03_models와 같은 계산)."""
    X_filled = X.fillna("WT")
    return pd.concat([binarize_mutations(X_filled),
                      mutation_count(X_filled),
                      mutation_type_counts(X_filled)], axis=1).astype("float32")

XGB_1109_PARAMS = dict(n_estimators=300, max_depth=5, learning_rate=0.05,
                       subsample=0.9, colsample_bytree=0.9,
                       objective="multi:softprob", eval_metric="mlogloss",
                       tree_method="hist", random_state=SEED, n_jobs=-1)

heavy_model = Pipeline([
    ("features", FunctionTransformer(derived_features)),
    ("model", XGBClassifier(**XGB_1109_PARAMS)),
]).fit(X, y)

heavy_result = measure(heavy_model, X_timing)
heavy_result

{'파일 크기(MB)': 11.0, '1명 예측(ms)': 746.74, '500명 예측(ms)': 1179.7}

## 5. 경량 vs 고성능 비교 (기획서 4장 표)
CV Macro F1은 팀 기록표 값(기존 5-fold 기준), 크기·속도는 위에서 같은 컴퓨터로 잰 값이에요.

In [11]:
comparison = pd.DataFrame([
    {"모델": "고성능: ③ 최신 XGBoost", "CV Macro F1": 0.4183, **heavy_result},
        {"모델": "경량: 로지스틱 회귀 (④)", "CV Macro F1": 0.4089, **light_result},
])
print("측정 환경:", platform.platform(), "| python", platform.python_version())
print("크기:", round(heavy_result["파일 크기(MB)"] / light_result["파일 크기(MB)"], 1), "배 차이")
print("1명 예측 속도:", round(heavy_result["1명 예측(ms)"] / light_result["1명 예측(ms)"], 1), "배 차이")
comparison

측정 환경: macOS-27.0.1-arm64-arm-64bit-Mach-O | python 3.13.18
크기: 23.4 배 차이
1명 예측 속도: 2.4 배 차이


,모델,CV Macro F1,파일 크기(MB),1명 예측(ms),500명 예측(ms)
0,고성능: ③ 최신 XGBoost,0.4183,11.00,746.74,1179.7
1,경량: 로지스틱 회귀 (④),0.4089,0.47,307.57,416.0


## 6. 시간 나눠 보기 — 전처리 vs 모델 예측
1명 예측 시간 중 글자→0/1 변환(전처리)과 모델 계산이 각각 얼마나 걸리는지 나눠서 재요.

In [10]:
def measure_split(model, X_sample, n_repeat=20):
    """1명 기준: 전처리 시간과 모델 예측 시간을 나눠 측정 (중앙값, ms)."""
    features, clf = model[:-1], model[-1]
    one = X_sample.iloc[:1]
    one_feat = features.transform(one)
    clf.predict_proba(one_feat)                      # 워밍업
    t_feat, t_model = [], []
    for _ in range(n_repeat):
        start = time.perf_counter()
        features.transform(one)
        t_feat.append((time.perf_counter() - start) * 1000)
        start = time.perf_counter()
        clf.predict_proba(one_feat)
        t_model.append((time.perf_counter() - start) * 1000)
    return {"전처리(ms)": round(float(np.median(t_feat)), 2),
            "모델 예측(ms)": round(float(np.median(t_model)), 3)}

print("경량  :", measure_split(light_model, X_timing))
print("고성능:", measure_split(heavy_model, X_timing))

경량  : {'전처리(ms)': 334.61, '모델 예측(ms)': 0.212}
고성능: {'전처리(ms)': 678.62, '모델 예측(ms)': 68.449}
